# YouTube merge (Google Drive + Colab)

No web UI. Edit the **config** cell, then run all cells.

1. Install / check `ffmpeg`
2. Mount Google Drive
3. Set paths + loop count / duration (supports multiple short clips)
4. Process and save the MP4 back to Drive

Upload `merge_engine.py` next to this notebook (same Drive folder), or upload the whole `colab/` folder to Drive and open this notebook from there.


## 1. Install ffmpeg

In [ ]:
import shutil
import subprocess

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg

print(subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])
print("ffmpeg ready")


## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted at /content/drive")


## 3. Config (edit this cell)

Use **either** `VIDEO_PATH` / `VIDEO_PATHS` **or** `IMAGE_PATH` (leave unused ones empty).

- `VIDEO_PATHS`: list of clips. With `STITCH = False` they are staged and the sequence loops to fit the audio. With `STITCH = True` they are joined end-to-end **once** (no looping).
- `KEEP_SOURCE_AUDIO`: stitch only - keep each clip's own audio. If `AUDIO_PATH` is set, it becomes the soundtrack instead.
- `VIDEO_PATH`: single clip (leave `VIDEO_PATHS` as `[]`).
- Paths are under `/content/drive/MyDrive/...` after mount.
- Duration **or** loop count (duration wins if both are set).
- Single video + audio: set `LOOP_MODE` to `"video"`, `"audio"`, or `"both"`.


In [ ]:
# --- EDIT THESE ---
VIDEO_PATH = ""  # single clip, e.g. "/content/drive/MyDrive/YT/inputs/clip.mp4"
VIDEO_PATHS = [
    # "/content/drive/MyDrive/YT/inputs/anim1.mp4",
    # "/content/drive/MyDrive/YT/inputs/anim2.mp4",
    # "/content/drive/MyDrive/YT/inputs/anim3.mp4",
]  # multiple clips
IMAGE_PATH = ""  # e.g. "/content/drive/MyDrive/YT/inputs/cover.png"
AUDIO_PATH = "/content/drive/MyDrive/YT/inputs/audio.mp3"

# Multi-clip behaviour (needs 2+ VIDEO_PATHS):
#   STITCH = False -> stage clips and loop the sequence to fit the audio.
#   STITCH = True  -> join clips end-to-end ONCE (no looping).
STITCH = False
KEEP_SOURCE_AUDIO = True  # stitch only: keep each clip's own audio (ignored if AUDIO_PATH is set)

LOOP_MODE = "both"  # video | audio | both  (single video + audio only)
LOOP_COUNT = None  # e.g. 108, or None
DURATION_MINUTES = None  # e.g. 5.0, or None

OUTPUT_FOLDER = "/content/drive/MyDrive/YTVideos"
OUTPUT_NAME = "merged.mp4"

# Folder that contains merge_engine.py (same folder as this notebook is fine)
ENGINE_DIR = "/content/drive/MyDrive/YT/colab"
# Temporary work folder (Colab local disk is faster than Drive)
WORK_DIR = "/content/yt_merge_tmp"
# --- END EDIT ---


## 4. Load merge engine

In [ ]:
import sys
from pathlib import Path

engine_dir = Path(ENGINE_DIR)
engine_file = engine_dir / "merge_engine.py"
if not engine_file.exists():
    raise FileNotFoundError(
        f"merge_engine.py not found at {engine_file}. "
        "Upload it to Drive and set ENGINE_DIR to that folder."
    )

sys.path.insert(0, str(engine_dir))
import merge_engine
import importlib

importlib.reload(merge_engine)
from merge_engine import run_merge

print("Loaded", engine_file)


## 5. Run merge → save to Drive

In [ ]:
from pathlib import Path

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

paths = [p for p in VIDEO_PATHS if str(p).strip()]
single = VIDEO_PATH.strip() or None
if paths and single:
    raise ValueError("Set either VIDEO_PATH or VIDEO_PATHS, not both.")
if STITCH and len(paths) < 2:
    raise ValueError("STITCH needs at least two VIDEO_PATHS.")

print("Starting merge… long jobs can take a while; leave this cell running.")
result = run_merge(
    video_path=None if paths else single,
    video_paths=paths or None,
    image_path=IMAGE_PATH or None,
    audio_path=AUDIO_PATH or None,
    output_folder=OUTPUT_FOLDER,
    output_name=OUTPUT_NAME,
    loop_mode=LOOP_MODE or None,
    loop_count=LOOP_COUNT,
    duration_minutes=DURATION_MINUTES,
    stitch=STITCH,
    keep_source_audio=KEEP_SOURCE_AUDIO,
    work_dir=WORK_DIR,
)

mins = result["duration_seconds"] / 60
print("Done.")
print(f"Saved: {result['path']}")
print(f"Duration: {result['duration_seconds']}s ({mins:.2f} min)")
print(f"Loop mode: {result['loop_mode']} | loop count: {result['loop_count']}")
